# Step 4a: Export tables for Power BI
Writes clean CSV files to `powerbi/data/`. Power BI will load these and do the visuals.

In [1]:
import duckdb
from pathlib import Path

BASE = Path.home() / "Desktop/projects/claims_analysis"
OUT = BASE / "powerbi/data"
OUT.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(BASE / "data/claims.duckdb"))

def export(name, sql):
    path = OUT / f"{name}.csv"
    con.execute(f"COPY ({sql}) TO '{path}' (HEADER, DELIMITER ',')")
    rows = con.execute(f"SELECT COUNT(*) FROM ({sql})").fetchall()[0][0]
    print(f"{name}.csv  {rows:,} rows")

## 1. Member-year table (the main table)
One row per member per year: demographics, months covered, cost by setting, chronic condition flags. PMPM and prevalence are calculated from this in Power BI.

In [2]:
conditions = {"SP_ALZHDMTA": "alzheimers", "SP_CHF": "heart_failure", "SP_CHRNKIDN": "kidney_disease",
              "SP_CNCR": "cancer", "SP_COPD": "copd", "SP_DEPRESSN": "depression", "SP_DIABETES": "diabetes",
              "SP_ISCHMCHT": "ischemic_heart", "SP_OSTEOPRS": "osteoporosis", "SP_RA_OA": "arthritis",
              "SP_STRKETIA": "stroke"}
flags = ", ".join(f"(m.{col} = 1)::INT AS {name}" for col, name in conditions.items())

export("member_year", f"""
SELECT my.DESYNPUF_ID AS member_id, my.year, my.age,
    CASE WHEN my.age < 65 THEN 'Under 65' WHEN my.age < 75 THEN '65-74'
         WHEN my.age < 85 THEN '75-84' ELSE '85+' END AS age_group,
    my.sex, my.state, my.member_months,
    my.ip_paid, my.op_paid, my.prof_paid, my.rx_paid, my.total_paid,
    {flags},
    {" + ".join(f"(m.{c} = 1)::INT" for c in conditions)} AS condition_count
FROM member_year my
JOIN members m ON m.DESYNPUF_ID = my.DESYNPUF_ID AND m.year = my.year
""")

member_year.csv  343,858 rows


## 2. Hospital stays
One row per stay with diagnosis name, length of stay, and a 30-day readmission flag.

In [3]:
export("stays", """
SELECT r.DESYNPUF_ID AS member_id, r.CLM_ID AS stay_id,
    r.admit_dt, r.discharge_dt, year(r.admit_dt) AS year,
    date_diff('day', r.admit_dt, r.discharge_dt) AS length_of_stay,
    r.drg, r.principal_dx, d.description AS principal_dx_name,
    r.paid,
    COALESCE(r.days_to_next BETWEEN 0 AND 30, false)::INT AS readmit_30d,
    (r.discharge_dt <= DATE '2010-12-01')::INT AS readmit_eligible
FROM readmits r
LEFT JOIN icd9_dx d ON r.principal_dx = d.code
""")

stays.csv  66,434 rows


## 3. Doctor services (CPT/HCPCS) by year

In [4]:
export("services", """
SELECT hcpcs, year, COUNT(*) AS services, SUM(paid) AS paid
FROM carrier_lines GROUP BY hcpcs, year
""")

services.csv  20,437 rows


## 4. Inpatient procedures (ICD-9) by year

In [5]:
export("procedures", """
SELECT i.ICD9_PRCDR_CD_1 AS icd9_proc, p.description, year(i.CLM_FROM_DT) AS year,
    COUNT(*) AS claims, SUM(i.CLM_PMT_AMT) AS paid
FROM inpatient i LEFT JOIN icd9_proc p ON i.ICD9_PRCDR_CD_1 = p.code
WHERE i.ICD9_PRCDR_CD_1 IS NOT NULL
GROUP BY ALL
""")

procedures.csv  2,753 rows


## 5. Risk scores

In [6]:
export("risk_scores", """
SELECT DESYNPUF_ID AS member_id, age, female, cost_2008, cost_2009,
    high_cost, risk_score, risk_tier
FROM risk_scores
""")

risk_scores.csv  114,618 rows


## 6. Check the files

In [7]:
sorted((p.name, f"{p.stat().st_size / 1e6:.1f} MB") for p in OUT.glob("*.csv"))

[('member_year.csv', '32.6 MB'),
 ('procedures.csv', '0.2 MB'),
 ('risk_scores.csv', '7.1 MB'),
 ('services.csv', '0.4 MB'),
 ('stays.csv', '8.5 MB')]